# Notebook 5: Ablation Study

## What we are ablating and why

An ablation study removes or changes one component at a time to show which design choices actually matter. Without this, an examiner can always ask: *"Would a simpler model have worked just as well?"*

We test three dimensions:

### Ablation 1 — Context window length (L)
How many past price steps does the encoder need to infer variance?
- L=5 (very short memory)
- L=10 (our trained model)
- L=20 (longer memory)

**Expected finding:** Heston variance is mean-reverting with half-life ~1/κ ≈ 1/1.5 ≈ 0.67 years = ~8 trading days. So L=10 should capture most of the signal; L=20 may add little.

### Ablation 2 — Embedding dimension (d_model)
How much capacity does the encoder need?
- d=16 (very small)
- d=32
- d=64 (our trained model)
- d=128 (larger)

**Expected finding:** d=64 should be close to optimal. d=16 may underfit; d=128 may overfit with small probe data.

### Ablation 3 — Observation mode
What if we give JEPA the true variance during training (oracle mode)?
- `prices` mode: input = log returns only (our model)
- `oracle` mode: input = [log returns, V_t]

**Expected finding:** oracle mode should give higher R² (upper bound), confirming that the gap in prices mode reflects genuine information loss from not observing V_t.

---
**Note:** Ablations 1 and 2 require retraining JEPA from scratch for each configuration (~5 min each on GPU). Ablation 3 uses a fresh JEPA trained in oracle mode. Run this notebook after Notebook 2 has completed.

## 0. Imports and shared setup

In [1]:
import sys, os, copy, time
sys.path.insert(0, os.path.join('..'))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

from heston_sim.generate import load_split
from jepa.dataset        import HestonWindowDataset
from jepa.model          import HestonJEPA

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
DATA_DIR = '../data'
SAVE_DIR = '../results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Device: {DEVICE}')

C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


Device: cuda


In [2]:
# ── Load data once, reuse across all ablations ─────────────────────────────
print('Loading data ...')
train_data = load_split('train', DATA_DIR)
val_data   = load_split('val',   DATA_DIR)
test_data  = load_split('test',  DATA_DIR)

# Subsample train for fast probe fitting
sub_idx = np.random.default_rng(42).choice(
    train_data['log_returns'].shape[0], size=50_000, replace=False)
train_sub = {k: v[sub_idx] for k, v in train_data.items()}
print(f'Train sub: {sub_idx.shape[0]:,} paths')

Loading data ...
Train sub: 50,000 paths


In [3]:
def train_jepa_config(context_len, target_len, d_model, mode,
                      n_epochs=30, batch_size=512, lr=3e-4,
                      n_heads=4, n_layers=2, label=''):
    """
    Train a fresh JEPA model from scratch for a given configuration.
    Returns the trained model.
    """
    feature_dim = 1 if mode == 'prices' else 2
    ds_tr = HestonWindowDataset(train_sub,  context_len, target_len, mode)
    ds_va = HestonWindowDataset(
        {k: v[np.random.default_rng(99).choice(
            val_data['log_returns'].shape[0], size=10_000, replace=False)]
         for k, v in val_data.items()},
        context_len, target_len, mode)

    ld_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  num_workers=0)
    ld_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0)

    model = HestonJEPA(feature_dim=feature_dim, d_model=d_model,
                       n_heads=min(n_heads, d_model//8) if d_model < 32 else n_heads,
                       n_layers=n_layers).to(DEVICE)
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    best_val, best_state = float('inf'), None
    pbar = tqdm(range(1, n_epochs+1), desc=f'  JEPA {label}', leave=False)
    for epoch in pbar:
        model.train()
        for batch in ld_tr:
            ctx = batch['context'].to(DEVICE)
            tgt = batch['target'].to(DEVICE)
            opt.zero_grad()
            loss, _, _ = model(ctx, tgt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            model.ema_update()
        sch.step()
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in ld_va:
                l, _, _ = model(batch['context'].to(DEVICE), batch['target'].to(DEVICE))
                val_losses.append(l.item())
        val_loss = np.mean(val_losses)
        if val_loss < best_val:
            best_val  = val_loss
            best_state = copy.deepcopy(model.state_dict())
        pbar.set_postfix({'val': f'{val_loss:.4f}'})

    model.load_state_dict(best_state)
    return model, best_val


def probe_r2(model, context_len, target_len, mode, n_probe=30_000):
    """
    Fit linear probe on train embeddings, evaluate R² on test.
    Returns r2, mae.
    """
    # Train embeddings
    ds = HestonWindowDataset(
        {k: v[np.random.default_rng(7).choice(
            train_sub['log_returns'].shape[0], size=min(n_probe, len(train_sub['log_returns'])), replace=False)]
         for k, v in train_sub.items()},
        context_len, target_len, mode)
    ld = DataLoader(ds, batch_size=1024, shuffle=False, num_workers=0)
    Z_tr, V_tr = [], []
    model.eval()
    with torch.no_grad():
        for b in ld:
            Z_tr.append(model.encode(b['context'].to(DEVICE)).cpu().numpy())
            V_tr.append(b['target_variance'].numpy())
    Z_tr = np.concatenate(Z_tr); V_tr = np.concatenate(V_tr)

    scaler = StandardScaler()
    ridge  = Ridge(alpha=1.0)
    ridge.fit(scaler.fit_transform(Z_tr), V_tr)

    # Test embeddings
    ds_te = HestonWindowDataset(test_data, context_len, target_len, mode)
    ld_te = DataLoader(ds_te, batch_size=1024, shuffle=False, num_workers=0)
    Z_te, V_te = [], []
    with torch.no_grad():
        for b in ld_te:
            Z_te.append(model.encode(b['context'].to(DEVICE)).cpu().numpy())
            V_te.append(b['target_variance'].numpy())
    Z_te = np.concatenate(Z_te); V_te = np.concatenate(V_te)

    V_pred = ridge.predict(scaler.transform(Z_te))
    return r2_score(V_te, V_pred), mean_absolute_error(V_te, V_pred)


print('Helper functions defined.')

Helper functions defined.


## Ablation 1: Context window length L

In [4]:
CONTEXT_LENS = [5, 10, 20]
TARGET_LEN   = 5
abl1_results = []

# Baseline: our trained model at L=10
print('Loading trained model (L=10) ...')
ckpt   = torch.load('../checkpoints/jepa_prices.pt', map_location=DEVICE, weights_only=False)
config = ckpt['config']
jepa_base = HestonJEPA(
    feature_dim=config['feature_dim'], d_model=config['d_model'],
    n_heads=config['n_heads'],         n_layers=config['n_layers'],
    ema_momentum=config['ema_momentum'], var_reg_coeff=config['var_reg_coeff']
).to(DEVICE)
jepa_base.load_state_dict(ckpt['model_state'])
r2_base, mae_base = probe_r2(jepa_base, 10, TARGET_LEN, 'prices')
abl1_results.append({'L': 10, 'r2': r2_base, 'mae': mae_base,
                     'val_loss': ckpt['val_loss'], 'note': 'pretrained'})
print(f'  L=10 (pretrained)  R²={r2_base:.4f}  MAE={mae_base:.5f}')

for L in [l for l in CONTEXT_LENS if l != 10]:
    print(f'\nTraining JEPA with L={L} ...')
    model, val_loss = train_jepa_config(
        context_len=L, target_len=TARGET_LEN,
        d_model=64, mode='prices', label=f'L={L}')
    r2, mae = probe_r2(model, L, TARGET_LEN, 'prices')
    abl1_results.append({'L': L, 'r2': r2, 'mae': mae, 'val_loss': val_loss})
    print(f'  L={L}  R²={r2:.4f}  MAE={mae:.5f}  val_loss={val_loss:.4f}')

abl1_results = sorted(abl1_results, key=lambda x: x['L'])
print('\nAblation 1 complete.')

Loading trained model (L=10) ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  L=10 (pretrained)  R²=0.7524  MAE=0.02158

Training JEPA with L=5 ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  JEPA L=5:   0%|          | 0/30 [00:00<?, ?it/s]

  L=5  R²=0.6861  MAE=0.02330  val_loss=0.6570

Training JEPA with L=20 ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  JEPA L=20:   0%|          | 0/30 [00:00<?, ?it/s]

C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.59276e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  L=20  R²=0.7929  MAE=0.02106  val_loss=0.0703

Ablation 1 complete.


## Ablation 2: Embedding dimension d_model

In [5]:
D_MODELS   = [16, 32, 64, 128]
abl2_results = []

# d=64 is already trained
abl2_results.append({'d': 64, 'r2': r2_base, 'mae': mae_base,
                     'val_loss': ckpt['val_loss'], 'note': 'pretrained'})
print(f'd=64 (pretrained)  R²={r2_base:.4f}  MAE={mae_base:.5f}')

for d in [dm for dm in D_MODELS if dm != 64]:
    # n_heads must divide d_model; use 2 heads for small dims
    n_heads = 2 if d < 32 else 4
    print(f'\nTraining JEPA with d={d}, n_heads={n_heads} ...')
    model, val_loss = train_jepa_config(
        context_len=10, target_len=5,
        d_model=d, mode='prices',
        n_heads=n_heads, label=f'd={d}')
    r2, mae = probe_r2(model, 10, 5, 'prices')
    abl2_results.append({'d': d, 'r2': r2, 'mae': mae, 'val_loss': val_loss})
    print(f'  d={d}  R²={r2:.4f}  MAE={mae:.5f}  val_loss={val_loss:.4f}')

abl2_results = sorted(abl2_results, key=lambda x: x['d'])
print('\nAblation 2 complete.')

d=64 (pretrained)  R²=0.7524  MAE=0.02158

Training JEPA with d=16, n_heads=2 ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  JEPA d=16:   0%|          | 0/30 [00:00<?, ?it/s]

  d=16  R²=0.7258  MAE=0.02284  val_loss=0.1454

Training JEPA with d=32, n_heads=4 ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  JEPA d=32:   0%|          | 0/30 [00:00<?, ?it/s]

C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.25644e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  d=32  R²=0.7682  MAE=0.02011  val_loss=0.7401

Training JEPA with d=128, n_heads=4 ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  JEPA d=128:   0%|          | 0/30 [00:00<?, ?it/s]

  d=128  R²=0.7231  MAE=0.02410  val_loss=0.3838

Ablation 2 complete.


## Ablation 3: Observation mode — prices only vs oracle (prices + true V_t)

In [ ]:
print('Training JEPA in oracle mode (input = [log returns, V_t]) ...')
jepa_oracle_mode, val_loss_oracle = train_jepa_config(
    context_len=10, target_len=5,
    d_model=64, mode='oracle', label='oracle mode')
r2_oracle_mode, mae_oracle_mode = probe_r2(jepa_oracle_mode, 10, 5, 'oracle')

abl3_results = [
    {'mode': 'prices (ours)',   'r2': r2_base,         'mae': mae_base,         'val_loss': ckpt['val_loss']},
    {'mode': 'oracle (V_t seen)', 'r2': r2_oracle_mode, 'mae': mae_oracle_mode, 'val_loss': val_loss_oracle},
]

print(f'\nPrices mode   R²={r2_base:.4f}  MAE={mae_base:.5f}')
print(f'Oracle mode   R²={r2_oracle_mode:.4f}  MAE={mae_oracle_mode:.5f}')
info_loss = r2_oracle_mode - r2_base
print(f'Information loss from not observing V_t: Δ R²={info_loss:.4f}')

Training JEPA in oracle mode (input = [log returns, V_t]) ...


c:\Users\USER\OneDrive\Desktop\IIT\Thesis\Implementation\Git Repo\JEPA---Deep-Hedging\Vibe coded experiment 2\Jepa-Heston\notebooks\..\jepa\encoder.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  JEPA oracle mode:   0%|          | 0/30 [00:00<?, ?it/s]

## Plot all ablation results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Ablation 1: Context length ─────────────────────────────────────────────
ax = axes[0]
ls = sorted(abl1_results, key=lambda x: x['L'])
Ls   = [r['L']  for r in ls]
r2s  = [r['r2'] for r in ls]
bars = ax.bar([str(l) for l in Ls], r2s, color='#3498db', width=0.5, edgecolor='white')
ax.axhline(r2_base, color='red', linestyle='--', lw=1.5, label=f'Our model (L=10): {r2_base:.3f}')
for bar, val in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Context window length L (steps)')
ax.set_ylabel('Linear probe R²  (V_t recovery)')
ax.set_title('Ablation 1: Context Length\nDoes longer memory help?')
ax.set_ylim(0, 1.0); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── Ablation 2: Embedding dimension ───────────────────────────────────────
ax = axes[1]
ds2 = sorted(abl2_results, key=lambda x: x['d'])
dims = [r['d']  for r in ds2]
r2s2 = [r['r2'] for r in ds2]
bars2 = ax.bar([str(d) for d in dims], r2s2, color='#2ecc71', width=0.5, edgecolor='white')
ax.axhline(r2_base, color='red', linestyle='--', lw=1.5, label=f'Our model (d=64): {r2_base:.3f}')
for bar, val in zip(bars2, r2s2):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Embedding dimension d_model')
ax.set_ylabel('Linear probe R²')
ax.set_title('Ablation 2: Embedding Dimension\nHow much capacity is needed?')
ax.set_ylim(0, 1.0); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── Ablation 3: Observation mode ───────────────────────────────────────────
ax = axes[2]
modes  = [r['mode'] for r in abl3_results]
r2s3   = [r['r2']   for r in abl3_results]
colors3 = ['#3498db', '#e74c3c']
bars3 = ax.bar(modes, r2s3, color=colors3, width=0.4, edgecolor='white')
for bar, val in zip(bars3, r2s3):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Linear probe R²')
ax.set_title('Ablation 3: Observation Mode\nCost of not observing V_t directly')
ax.set_ylim(0, 1.05)
diff = r2s3[1] - r2s3[0]
ax.annotate(f'Δ R² = +{diff:.3f}\n(information gain\nfrom seeing V_t)',
            xy=(1, r2s3[1]), xytext=(0.55, r2s3[1] - 0.12),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('JEPA Ablation Study: Which design choices matter for volatility recovery?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {SAVE_DIR}/ablation_study.png')

## Summary table

In [ ]:
print('='*60)
print('ABLATION SUMMARY')
print('='*60)

print('\nAblation 1 — Context window length (d=64, prices mode)')
print(f'{"L":>5}  {"R²":>7}  {"MAE":>9}')
for r in sorted(abl1_results, key=lambda x: x['L']):
    marker = ' ◄ our model' if r['L'] == 10 else ''
    print(f'{r["L"]:>5}  {r["r2"]:>7.4f}  {r["mae"]:>9.5f}{marker}')

print('\nAblation 2 — Embedding dimension (L=10, prices mode)')
print(f'{"d":>5}  {"R²":>7}  {"MAE":>9}')
for r in sorted(abl2_results, key=lambda x: x['d']):
    marker = ' ◄ our model' if r['d'] == 64 else ''
    print(f'{r["d"]:>5}  {r["r2"]:>7.4f}  {r["mae"]:>9.5f}{marker}')

print('\nAblation 3 — Observation mode (L=10, d=64)')
print(f'{"Mode":<25}  {"R²":>7}  {"MAE":>9}')
for r in abl3_results:
    print(f'{r["mode"]:<25}  {r["r2"]:>7.4f}  {r["mae"]:>9.5f}')

print('\nKey finding: prices mode recovers R²={:.3f} of V_t,'.format(r2_base))
print(f'vs oracle upper bound R²={r2_oracle_mode:.3f}.')
print(f'The {info_loss:.3f} R² gap is the irreducible information loss')
print('from not directly observing the latent variance process.')

np.savez(f'{SAVE_DIR}/ablation_metrics.npz',
         abl1_L    = [r['L']  for r in sorted(abl1_results, key=lambda x: x['L'])],
         abl1_r2   = [r['r2'] for r in sorted(abl1_results, key=lambda x: x['L'])],
         abl2_d    = [r['d']  for r in sorted(abl2_results, key=lambda x: x['d'])],
         abl2_r2   = [r['r2'] for r in sorted(abl2_results, key=lambda x: x['d'])],
         prices_r2 = r2_base,
         oracle_r2 = r2_oracle_mode)
print(f'\nSaved → {SAVE_DIR}/ablation_metrics.npz')